In [3]:
import numpy as np
import random
import math
# Levy flight using Mantegna's algorithm
def levy_flight():
    beta = 1.5
    sigma = (
        math.gamma(1 + beta) * math.sin(math.pi * beta / 2) /
        (math.gamma((1 + beta) / 2) * beta *
         2 ** ((beta - 1) / 2))
    ) ** (1 / beta)

    u = np.random.normal(0, sigma)
    v = np.random.normal(0, 1)

    return u / (abs(v) ** (1 / beta) + 1e-10)
# Convert continuous solution into binary station selection
def convert_to_binary(solution):
    probability = 1 / (1 + np.exp(-np.clip(solution, -20, 20)))
    binary = np.array([
        1 if random.random() < p else 0
        for p in probability
    ])
    # At least one charging station must be selected
    if np.sum(binary) == 0:
        binary[np.argmax(probability)] = 1
    return binary
# Calculate fitness of a nest
def fitness(solution, installation_cost, operating_cost,
            demand, distances, service_distance,
            weights):

    selected = np.where(solution == 1)[0]

    total_installation = np.sum(installation_cost[selected])
    total_operating = np.sum(operating_cost[selected])

    # Find nearest selected station for each user
    user_distances = distances[:, selected]
    nearest_distances = np.min(user_distances, axis=1)

    # Demand-weighted travel distance
    total_demand = np.sum(demand)

    average_distance = (
        np.sum(nearest_distances * demand) / total_demand
    )

    # Calculate demand coverage
    covered = nearest_distances <= service_distance

    coverage = (
        np.sum(demand[covered]) / total_demand
    ) * 100

    # Normalize objective components
    cost_scale = np.sum(installation_cost + operating_cost)
    cost_scale = max(cost_scale, 1)

    distance_scale = max(np.max(distances), 1)

    normalized_cost = (
        total_installation + total_operating
    ) / cost_scale

    normalized_distance = average_distance / distance_scale

    uncovered = 1 - coverage / 100

    # Minimize the fitness value
    score = (
        weights[0] * normalized_cost +
        weights[1] * normalized_distance +
        weights[2] * uncovered
    )

    return score, total_installation, total_operating, \
        average_distance, coverage


# Cuckoo Search Algorithm
def cuckoo_search(installation_cost, operating_cost,
                  demand, distances, service_distance,
                  nests_count, max_iterations,
                  discovery_probability, weights):

    locations = len(installation_cost)

    # Initialize nests randomly
    nests = np.random.uniform(-2, 2, (nests_count, locations))

    best_solution = None
    best_fitness = float('inf')

    for iteration in range(max_iterations):

        # Generate new solutions using Levy flight
        for i in range(nests_count):

            new_nest = nests[i] + levy_flight() * np.random.normal(
                0, 1, locations
            )

            new_solution = convert_to_binary(new_nest)

            selected_nest = random.randrange(nests_count)
            current_solution = convert_to_binary(
                nests[selected_nest]
            )

            new_score = fitness(
                new_solution, installation_cost,
                operating_cost, demand, distances,
                service_distance, weights
            )[0]

            old_score = fitness(
                current_solution, installation_cost,
                operating_cost, demand, distances,
                service_distance, weights
            )[0]

            if new_score < old_score:
                nests[selected_nest] = new_nest

        # Abandon a fraction of nests
        abandon_count = int(
            discovery_probability * nests_count
        )

        if abandon_count > 0:
            scores = []

            for nest in nests:
                binary = convert_to_binary(nest)

                score = fitness(
                    binary, installation_cost,
                    operating_cost, demand, distances,
                    service_distance, weights
                )[0]

                scores.append(score)

            worst_indices = np.argsort(scores)[-abandon_count:]

            for index in worst_indices:
                nests[index] = np.random.uniform(
                    -2, 2, locations
                )

        # Find the best nest
        for nest in nests:
            solution = convert_to_binary(nest)

            result = fitness(
                solution, installation_cost,
                operating_cost, demand, distances,
                service_distance, weights
            )

            if result[0] < best_fitness:
                best_fitness = result[0]
                best_solution = solution.copy()

    return best_solution


# Main program
def main():

    print("EV CHARGING STATION PLACEMENT")
    print("Using Cuckoo Search Algorithm")

    # Read input parameters
    n = int(input("\nEnter number of candidate locations: "))
    m = int(input("Enter number of EV users: "))

    installation_cost = np.array([
        float(input(f"Installation cost of location {i+1}: "))
        for i in range(n)
    ])

    operating_cost = np.array([
        float(input(f"Operating cost of location {i+1}: "))
        for i in range(n)
    ])

    demand = np.array([
        float(input(f"Charging demand of user {i+1}: "))
        for i in range(m)
    ])

    # Read distance matrix
    distances = np.zeros((m, n))

    print("\nEnter distance from each user to each location:")

    for i in range(m):
        for j in range(n):
            distances[i][j] = float(
                input(f"User {i+1} to Location {j+1}: ")
            )

    service_distance = float(
        input("\nEnter maximum service distance: ")
    )

    nests_count = int(input("Enter number of nests: "))
    max_iterations = int(input("Enter maximum iterations: "))

    discovery_probability = float(
        input("Enter discovery probability (0 to 1): ")
    )

    print("\nEnter objective weights:")
    w1 = float(input("Installation and operating cost weight: "))
    w2 = float(input("Travel distance weight: "))
    w3 = float(input("Coverage weight: "))

    if (n <= 0 or m <= 0 or nests_count < 2 or
            max_iterations <= 0 or
            not 0 <= discovery_probability <= 1 or
            np.any(installation_cost < 0) or
            np.any(operating_cost < 0) or
            np.any(demand < 0) or
            np.any(distances < 0) or
            service_distance < 0 or
            np.sum(demand) <= 0 or
            min(w1, w2, w3) < 0 or
            w1 + w2 + w3 <= 0):

        print("\nInvalid input. Please check the values.")
        return

    weights = np.array([w1, w2, w3])
    weights = weights / np.sum(weights)

    # Run Cuckoo Search
    best_solution = cuckoo_search(
        installation_cost, operating_cost,
        demand, distances, service_distance,
        nests_count, max_iterations,
        discovery_probability, weights
    )

    # Display final result
    result = fitness(
        best_solution, installation_cost,
        operating_cost, demand, distances,
        service_distance, weights
    )

    print("\n------ OPTIMAL RESULT ------")

    selected = np.where(best_solution == 1)[0]

    print("\nSelected charging station locations:")

    for location in selected:
        print(f"Location {location+1}")

    print("\nTotal number of stations:", len(selected))
    print("Installation cost:", result[1])
    print("Operating cost:", result[2])
    print("Average user travel distance:", round(result[3], 2))
    print("Demand coverage:", round(result[4], 2), "%")
    print("Best fitness value:", round(result[0], 5))


if __name__ == "__main__":
    main()
0.

EV CHARGING STATION PLACEMENT
Using Cuckoo Search Algorithm

Enter number of candidate locations: 3
Enter number of EV users: 4
Installation cost of location 1: 500000
Installation cost of location 2: 800000
Installation cost of location 3: 600000
Operating cost of location 1: 50000
Operating cost of location 2: 40000
Operating cost of location 3: 45000
Charging demand of user 1: 10
Charging demand of user 2: 20
Charging demand of user 3: 15
Charging demand of user 4: 25

Enter distance from each user to each location:
User 1 to Location 1: 2
User 1 to Location 2: 8
User 1 to Location 3: 5
User 2 to Location 1: 7
User 2 to Location 2: 3
User 2 to Location 3: 6
User 3 to Location 1: 4
User 3 to Location 2: 6
User 3 to Location 3: 2
User 4 to Location 1: 9
User 4 to Location 2: 4
User 4 to Location 3: 7

Enter maximum service distance: 5
Enter number of nests: 20
Enter maximum iterations: 100
Enter discovery probability (0 to 1): 0.25

Enter objective weights:
Installation and operating 

0.0